In [1]:
import pickle, os
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_max_pool
from torch_geometric.data import Batch
from torch_geometric.loader import DataLoader
from sklearn import metrics
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

C:\Users\noton\miniconda3\envs\OPSdesign2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class Net(nn.Module):
    def __init__(self, in_dim=30, hidden=256, out_dim=6):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.conv3 = GCNConv(hidden, hidden)
        self.conv4 = GCNConv(hidden, hidden)
        self.fc1 = nn.Linear(hidden, hidden)
        self.fc2 = nn.Linear(hidden, out_dim)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = F.relu(self.conv3(x, edge_index))
        x = F.relu(self.conv4(x, edge_index))
        x = global_max_pool(x, data.batch)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [3]:
def to_numpy_1d(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy().astype(np.float64, copy=False).reshape(-1)
    if hasattr(x, "to_numpy"):
        return x.to_numpy(dtype=np.float64, copy=False).reshape(-1)
    return np.asarray(x, dtype=np.float64).reshape(-1)

def custom_collate(batch):
    data_list, target_list = zip(*batch)
    batch_data = Batch.from_data_list(list(data_list))
    batch_target = torch.stack(list(target_list))
    return batch_data, batch_target

In [4]:
torch.manual_seed(42)

#device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = torch.device('cpu')

targets = ["eHOMO", "eLUMO", "DE_S0S1", "f_S0S1", "DE_S0T1", "DES1T1"]
T = len(targets)

with open('data/data_train_list.pkl', 'rb') as f:
    data_train_list = pickle.load(f)
with open('data/data_val_list.pkl', 'rb') as f:
    data_val_list = pickle.load(f)
with open('data/data_test_list.pkl', 'rb') as f:
    data_test_list = pickle.load(f)

def load_targets(split):
    Y_cols = []
    for name in targets:
        with open(f'data/target_{split}_{name}.pkl', 'rb') as f:
            y = to_numpy_1d(pickle.load(f))
            Y_cols.append(y)
    Y = np.column_stack(Y_cols).astype(np.float64, copy=False)  # (N, T)
    return Y

y_tr_raw = load_targets("train")
y_va_raw = load_targets("val")
y_te_raw = load_targets("test")

scaler = StandardScaler()
y_tr_sc = scaler.fit_transform(y_tr_raw)
y_va_sc = scaler.transform(y_va_raw)
y_te_sc = scaler.transform(y_te_raw)

y_tr_t = torch.from_numpy(y_tr_sc).float()
y_va_t = torch.from_numpy(y_va_sc).float()
y_te_t = torch.from_numpy(y_te_sc).float()

train_ds = [(g, y_tr_t[i]) for i, g in enumerate(data_train_list)]
val_ds   = [(g, y_va_t[i]) for i, g in enumerate(data_val_list)]
test_ds  = [(g, y_te_t[i]) for i, g in enumerate(data_test_list)]

train_loader       = DataLoader(train_ds, batch_size=128, shuffle=True,  collate_fn=custom_collate)
val_loader         = DataLoader(val_ds,   batch_size=128, shuffle=False, collate_fn=custom_collate)
train_eval_loader  = DataLoader(train_ds, batch_size=128, shuffle=False, collate_fn=custom_collate)
test_loader        = DataLoader(test_ds,  batch_size=128, shuffle=False, collate_fn=custom_collate)

model = Net(in_dim=30, hidden=256, out_dim=T).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss(reduction='mean')

max_epochs = 100
patience   = 30
best_val = float('inf')
wait = 0

for epoch in range(max_epochs):
    model.train()
    for data, target in train_loader:
        data   = data.to(device)
        target = target.to(device)
        pred   = model(data)
        loss   = criterion(pred, target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    model.eval()
    val_sum = 0.0
    n_samp  = 0
    with torch.no_grad():
        for data, target in val_loader:
            data   = data.to(device)
            target = target.to(device)
            pred   = model(data)
            batch_loss = criterion(pred, target).item()
            bs = target.size(0)
            val_sum += batch_loss * bs
            n_samp  += bs
    val_loss = val_sum / max(n_samp, 1)
    print(f"Epoch {epoch+1:03d} | Val Loss: {val_loss:.6f}")

    if val_loss < best_val - 1e-9:
        best_val = val_loss
        wait = 0
        os.makedirs('GCNmodel', exist_ok=True)
        torch.save(model.state_dict(), 'GCNmodel/model_mt_sc.pth')
        with open('GCNmodel/scaler_mt.pkl', 'wb') as f:
            pickle.dump(scaler, f)
    else:
        wait += 1
        if wait >= patience:
            print(f'Early stopping at epoch {epoch+1} (best val={best_val:.6f})')
            break

model.load_state_dict(torch.load('GCNmodel/model_mt_sc.pth', map_location=device))
model.eval()

def predict_scaled(loader):
    preds = []
    with torch.no_grad():
        for data, _ in loader:
            out = model(data.to(device)).cpu().numpy().astype(np.float64, copy=False)
            preds.append(out)
    return np.concatenate(preds, axis=0)

pred_tr_sc = predict_scaled(train_eval_loader)
pred_te_sc = predict_scaled(test_loader)

pred_tr = scaler.inverse_transform(pred_tr_sc)
pred_te = scaler.inverse_transform(pred_te_sc)

print('=========================================')
for j, name in enumerate(targets):
    r2_tr  = metrics.r2_score(y_tr_raw[:, j], pred_tr[:, j])
    r2_te  = metrics.r2_score(y_te_raw[:, j], pred_te[:, j])
    rmse_tr = float(np.sqrt(metrics.mean_squared_error(y_tr_raw[:, j], pred_tr[:, j])))
    rmse_te = float(np.sqrt(metrics.mean_squared_error(y_te_raw[:, j], pred_te[:, j])))
    print(f'[{name}] R2 train: {r2_tr:.2f} | test: {r2_te:.2f} | RMSE train: {rmse_tr:.2f} | test: {rmse_te:.2f}')

mean_r2_tr = np.mean([metrics.r2_score(y_tr_raw[:, j], pred_tr[:, j]) for j in range(T)])
mean_r2_te = np.mean([metrics.r2_score(y_te_raw[:, j], pred_te[:, j]) for j in range(T)])
print(f'== Mean R2 train: {mean_r2_tr:.2f} | test: {mean_r2_te:.2f}')

Epoch 001 | Val Loss: 0.253439
Epoch 002 | Val Loss: 0.204337
Epoch 003 | Val Loss: 0.184083
Epoch 004 | Val Loss: 0.164242
Epoch 005 | Val Loss: 0.148267
Epoch 006 | Val Loss: 0.123324
Epoch 007 | Val Loss: 0.124977
Epoch 008 | Val Loss: 0.115061
Epoch 009 | Val Loss: 0.128852
Epoch 010 | Val Loss: 0.114053
Epoch 011 | Val Loss: 0.107202
Epoch 012 | Val Loss: 0.098921
Epoch 013 | Val Loss: 0.096267
Epoch 014 | Val Loss: 0.092567
Epoch 015 | Val Loss: 0.094911
Epoch 016 | Val Loss: 0.090922
Epoch 017 | Val Loss: 0.087490
Epoch 018 | Val Loss: 0.085543
Epoch 019 | Val Loss: 0.094777
Epoch 020 | Val Loss: 0.085151
Epoch 021 | Val Loss: 0.081896
Epoch 022 | Val Loss: 0.079162
Epoch 023 | Val Loss: 0.076897
Epoch 024 | Val Loss: 0.082107
Epoch 025 | Val Loss: 0.081056
Epoch 026 | Val Loss: 0.077472
Epoch 027 | Val Loss: 0.077433
Epoch 028 | Val Loss: 0.077720
Epoch 029 | Val Loss: 0.071201
Epoch 030 | Val Loss: 0.081589
Epoch 031 | Val Loss: 0.073150
Epoch 032 | Val Loss: 0.069737
Epoch 03